# Week 6: LoRA — Get 95% of the Accuracy with 1.6% of the Parameters
**TMI-102 | PaAC, IIT Roorkee**

---

### What you'll do this week
1. Learn what **LoRA** (Low-Rank Adaptation) is
2. See how it works with a simple analogy
3. Apply LoRA to ResNet-50 and compare with Weeks 4-5

### The Big Idea (no scary math needed!)
Instead of updating ALL 25.6M weights, LoRA adds tiny "adapter" matrices that learn the *difference* between ImageNet features and galaxy features. Think of it like adding Post-it notes to a textbook instead of rewriting the whole book.

### 📚 Read This
- [LoRA Paper (just read the Abstract + Section 1)](https://arxiv.org/abs/2106.09685)
- [Visual LoRA Explanation](https://magazine.sebastianraschka.com/p/lora-and-dora-from-scratch) — great diagrams

### Q1 (5 pts)
In your own words (2-3 sentences), explain the LoRA idea using an analogy. Example: "It's like ___"

> It's like having a textbook that you aren't allowed to write in, so instead of buying a whole new edition, you just stick a few sticky notes on the pages with the updated info. The original book stays totally untouched, but the sticky notes do all the extra work.

### Q2 (5 pts)
If the original weight matrix has 2048 × 512 = 1,048,576 parameters, and LoRA uses rank r=4, how many LoRA parameters are there? (Just multiply: 2048×4 + 4×512)

> 10240

In [9]:
# ── Setup ──
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np, matplotlib.pyplot as plt, time

DATASET_DIR = r'C:\Users\Lenovo\Documents\pranay 1\python 1.0\Galaxy_Dataset\dataset_split'
IMG_SIZE = 224; BATCH_SIZE = 32

datagen = ImageDataGenerator(rescale=1./255)
train_gen = datagen.flow_from_directory(f'{DATASET_DIR}/train', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical')
val_gen = datagen.flow_from_directory(f'{DATASET_DIR}/val', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical')
test_gen = datagen.flow_from_directory(f'{DATASET_DIR}/test', target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE, class_mode='categorical', shuffle=False)


Found 43104 images belonging to 3 classes.
Found 9234 images belonging to 3 classes.
Found 9240 images belonging to 3 classes.


### Task 1: Understand the LoRA Layer (10 pts)

This code implements a LoRA layer. **Read through it** — you don't need to write it, just understand what each part does.

In [11]:
class LoRADense(layers.Layer):
    """A Dense layer with a LoRA adapter bolted on.
    
    Instead of updating the big weight matrix W (frozen),
    we learn two small matrices A and B such that:
        output = W @ x + (B @ A @ x) * scaling
    
    Think of B @ A as a 'correction' to W.
    """
    def __init__(self, units, rank=4, alpha=32, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # = 8.0 for our settings
        self.activation_fn = tf.keras.activations.get(activation)
        
    def build(self, input_shape):
        input_dim = input_shape[-1]
        
        # Original weight (FROZEN — we won't update this)
        self.W = self.add_weight('kernel', (input_dim, self.units),
                                 initializer='glorot_uniform', trainable=False)
        self.b = self.add_weight('bias', (self.units,),
                                 initializer='zeros', trainable=False)
        
        # LoRA adapters (SMALL and TRAINABLE)
        self.A = self.add_weight('lora_A', (input_dim, self.rank),
                                 initializer='glorot_normal', trainable=True)
        self.B = self.add_weight('lora_B', (self.rank, self.units),
                                 initializer='zeros', trainable=True)
    
    def call(self, x):
        # Original path (frozen)
        h = tf.matmul(x, self.W) + self.b
        # LoRA correction path (trainable)
        h = h + tf.matmul(tf.matmul(x, self.A), self.B) * self.scaling
        return self.activation_fn(h) if self.activation_fn else h

# Quick test
test = LoRADense(512, rank=4)
test.build((None, 2048))
lora_params = 2048*4 + 4*512
orig_params = 2048*512
print(f"Original params: {orig_params:,}")
print(f"LoRA params:     {lora_params:,}")
print(f"Compression:     {lora_params/orig_params*100:.1f}% of original")
print(f"That's {orig_params/lora_params:.0f}× fewer parameters! 🎉")

ValueError: `name` must be passed as a keyword argument. Received: add_weight('kernel', ...). Use: add_weight(shape=..., name='kernel').

### Task 2: Build ResNet-50 with LoRA (15 pts)

The model is mostly pre-built. **Fill in the rank value** (try r=4).

In [16]:
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG_SIZE,IMG_SIZE,3))
for layer in base_model.layers:
    layer.trainable = False  # Freeze backbone

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)

# ╔══════════════════════════════════════════╗
# ║  TODO: Set the rank (try 4)              ║
# ╚══════════════════════════════════════════╝
x = LoRADense(256, rank=4, alpha=32, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model = Model(inputs, outputs)
trainable = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"🔓 Trainable: {trainable:,} params")
print(f"🔒 Total: {model.count_params():,} params")
print(f"📉 Only training {trainable/model.count_params()*100:.2f}% of the model!")


🔓 Trainable: 9,987 params
🔒 Total: 24,122,243 params
📉 Only training 0.04% of the model!


In [15]:
import tensorflow as tf
from tensorflow.keras import layers

class LoRADense(layers.Layer):
    def __init__(self, units, rank=4, alpha=32, activation=None, use_bias=True, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.rank = rank
        self.alpha = alpha
        self.activation_fn = tf.keras.activations.get(activation)
        self.use_bias = use_bias

    def build(self, input_shape):
        input_dim = input_shape[-1]

        # Original weight (FROZEN) - Keras 3 keyword arguments only
        self.W = self.add_weight(
            shape=(input_dim, self.units), 
            name='kernel', 
            initializer='glorot_uniform', 
            trainable=False
        )
        
        # Optional bias
        if self.use_bias:
            self.b = self.add_weight(
                shape=(self.units,), 
                name='bias', 
                initializer='zeros', 
                trainable=False
            )

        # LoRA matrices A and B (TRAINABLE)
        self.lora_A = self.add_weight(
            shape=(input_dim, self.rank), 
            name='lora_A', 
            initializer='he_normal', 
            trainable=True
        )
        self.lora_B = self.add_weight(
            shape=(self.rank, self.units), 
            name='lora_B', 
            initializer='zeros', 
            trainable=True
        )
        
        self.scale = self.alpha / self.rank
        self.built = True

    def call(self, inputs):
        # Base layer output using frozen weights
        orig_output = tf.matmul(inputs, self.W)
        
        # LoRA path: matmul(inputs, A) * B * scale
        lora_output = tf.matmul(tf.matmul(inputs, self.lora_A), self.lora_B) * self.scale
        
        output = orig_output + lora_output
        
        if self.use_bias:
            output = output + self.b
            
        return self.activation_fn(output) if self.activation_fn else output

### Task 3: Train & Compare (15 pts)

In [17]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), 
              loss='categorical_crossentropy', metrics=['accuracy'])

start = time.time()
history = model.fit(train_gen, validation_data=val_gen, epochs=12,
    callbacks=[EarlyStopping(patience=4, restore_best_weights=True)])
elapsed = time.time() - start

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'], label='Train'); ax1.plot(history.history['val_accuracy'], label='Val')
ax1.set_title('LoRA — Accuracy'); ax1.legend(); ax1.grid(True)
ax2.plot(history.history['loss'], label='Train'); ax2.plot(history.history['val_loss'], label='Val')
ax2.set_title('LoRA — Loss'); ax2.legend(); ax2.grid(True)
plt.tight_layout(); plt.show()

test_loss, test_acc = model.evaluate(test_gen)
print(f"\n🎯 LoRA Test Accuracy: {test_acc*100:.2f}%")
print(f"⏱️  Time: {elapsed:.0f}s total ({elapsed/len(history.history['loss']):.0f}s/epoch)")

Epoch 1/12
1347/1347 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5505 - loss: 0.6977

KeyboardInterrupt: 

### Final Comparison Table (10 pts)

Fill in with YOUR numbers:

| Model | Params Trained | Test Accuracy | Time/Epoch |
|:---|:---:|:---:|:---:|
| Week 4: Frozen | ~2K | % | s |
| Week 5: Full FT | ~25.6M | % | s |
| Week 6: LoRA (r=4) | ~???K | % | s |

### Q3 (5 pts)
Is the small accuracy drop from Full FT → LoRA worth the massive speedup and parameter savings? When would you use each?

> Yes, absolutely. LoRA achieves nearly identical accuracy to Full Fine-Tuning while training only a fraction of the parameters (reducing trainable parameters from millions to just thousands) and drastically cutting down training time.

When to use Full Fine-Tuning: When you have abundant computational resources, plenty of time, a massive dataset, and you need to squeeze out every last fraction of a percent in performance.

When to use LoRA: When you are working with limited hardware, constrained memory, or need to quickly adapt large models efficiently without high storage costs for multiple task-specific checkpoints.

---
**Submit as:** `Week_6_YourName.ipynb`